# 90 — Tox21 NR Panel: Fetch + Biological Fingerprint

The Tox21 dataset has 12 nuclear receptor / stress-response assays for ~8,000 compounds.
Key NR assays: NR-AhR, NR-AR, NR-AR-LBD, NR-ER, NR-ER-LBD, NR-PPAR-gamma.

AhR (aryl hydrocarbon receptor) is co-regulated with PXR for many xenobiotic compounds — shared CYP1A2/CYP3A4 induction pathway.

Strategy:
1. Download Tox21 via DeepChem or MoleculeNet
2. Train one LGBM binary classifier per assay (6 NR assays)
3. Predicted probabilities → 6-dim Tox21 biological fingerprint
4. Combine with ChEMBL bio-FP (nb87) → 11-dim biological fingerprint
5. Train PXR LGBM with combined + bio-FPs

In [1]:
import os, sys, warnings
os.environ["PYTHONIOENCODING"] = "utf-8"
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy import stats
from pathlib import Path
from pxr.data import load_train, load_test
from pxr.featurize import combined, impute
from pxr.eval import rae, scaffold_kfold_indices
from pxr.chem import bemis_murcko, morgan_fp_batch, standardize_smiles, compute_physchem
from pxr.paths import DATA_PROCESSED, DATA_EXTERNAL, SUBMISSIONS
SEED = 42; N_FOLDS = 5
LGBM = dict(n_estimators=1000, num_leaves=64, learning_rate=0.05,
            min_child_samples=10, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=0.1, random_state=SEED, verbose=-1, n_jobs=4)


In [2]:
def full_metrics(y_true, y_pred, cp=None, label=""):
    yt = np.asarray(y_true, float); yp = np.asarray(y_pred, float)
    msk = np.isfinite(yt) & np.isfinite(yp); yt, yp = yt[msk], yp[msk]
    mae = float(np.mean(np.abs(yt-yp)))
    rae_v = mae / float(np.mean(np.abs(yt-yt.mean()))) if yt.std()>0 else float("nan")
    r2  = 1-np.sum((yt-yp)**2)/np.sum((yt-yt.mean())**2) if yt.std()>0 else float("nan")
    pr, _ = stats.pearsonr(yt, yp); sp, _ = stats.spearmanr(yt, yp)
    kt, _ = stats.kendalltau(yt, yp)
    m = dict(RAE=rae_v, MAE=mae, R2=float(r2), Pearson=float(pr),
             Spearman=float(sp), Kendall=float(kt))
    if cp is not None and hasattr(cp, "iterrows") and len(cp) > 0:
        c=t=0
        for _,row in cp.iterrows():
            ia,ii = int(row.get("idx_active",-1)), int(row.get("idx_inactive",-1))
            if 0<=ia<len(yp) and 0<=ii<len(yp): c+=int(yp[ia]>yp[ii]); t+=1
        m["Cliff_acc"] = c/t if t else float("nan")
    if label:
        ca = f"  Cliff={m.get('Cliff_acc',float('nan')):.3f}" if "Cliff_acc" in m else ""
        print(f"  [{label}] RAE={rae_v:.4f} MAE={mae:.4f} R²={r2:.4f} "
              f"r={pr:.4f} ρ={sp:.4f} τ={kt:.4f}{ca}")
    return m


In [3]:
tr = load_train(); te = load_test()
y_tr = tr["pec50"].values.astype(np.float64)
scaffolds = tr["smiles"].map(bemis_murcko).tolist()
splits = scaffold_kfold_indices(scaffolds, N_FOLDS, SEED)
active_mask = y_tr >= 5.5
X_tr = impute(combined(tr["smiles"].tolist()))
X_te = impute(combined(te["smiles"].tolist()))
fps_tr = morgan_fp_batch(tr["smiles"].tolist()).astype(np.float32)
fps_te = morgan_fp_batch(te["smiles"].tolist()).astype(np.float32)
cliff_pairs = (pd.read_parquet(DATA_PROCESSED/"cliff_pairs.parquet")
               if (DATA_PROCESSED/"cliff_pairs.parquet").exists() else pd.DataFrame())
# Build idx_active / idx_inactive
if len(cliff_pairs) > 0:
    s2i = {s:i for i,s in enumerate(tr["smiles"].tolist())}
    ac = "cliff_active_smiles" if "cliff_active_smiles" in cliff_pairs.columns else "smiles_a"
    ic = "cliff_inactive_smiles" if "cliff_inactive_smiles" in cliff_pairs.columns else "smiles_b"
    cliff_pairs["idx_active"]   = cliff_pairs[ac].map(s2i)
    cliff_pairs["idx_inactive"] = cliff_pairs[ic].map(s2i)
    cliff_pairs = cliff_pairs.dropna(subset=["idx_active","idx_inactive"])
    cliff_pairs[["idx_active","idx_inactive"]] = cliff_pairs[["idx_active","idx_inactive"]].astype(int)
print(f"Train {len(tr):,}  Test {len(te):,}  Cliffs {len(cliff_pairs)}")


Train 4,139  Test 513  Cliffs 0


In [4]:
# Fetch Tox21 data
import urllib.request, io, zipfile, csv

TOX21_URL = "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/tox21.csv.gz"
print(f"Downloading Tox21 from {TOX21_URL}...", flush=True)
try:
    import gzip
    with urllib.request.urlopen(TOX21_URL, timeout=120) as resp:
        raw = gzip.decompress(resp.read()).decode("utf-8")
    tox21_df = pd.read_csv(io.StringIO(raw))
    print(f"Tox21: {len(tox21_df)} compounds, columns: {list(tox21_df.columns)}")
    tox21_df.to_parquet(DATA_EXTERNAL / "tox21_nr_data.parquet", index=False)
    print("Saved tox21_nr_data.parquet")
except Exception as e:
    print(f"Download failed: {e}  — trying DeepChem import")
    try:
        import deepchem as dc
        tasks, datasets, _ = dc.molnet.load_tox21()
        train_ds, val_ds, test_ds = datasets
        all_X = [train_ds.X, val_ds.X, test_ds.X]
        tox21_df = None
        print("DeepChem Tox21 loaded (featurized)")
    except:
        print("DeepChem not available — using empty DataFrame")
        tox21_df = pd.DataFrame()

TOX21_NR_TASKS = ["NR-AhR","NR-AR","NR-AR-LBD","NR-ER","NR-ER-LBD","NR-PPAR-gamma"]


Tox21: 7831 compounds, columns: ['NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER', 'NR-ER-LBD', 'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53', 'mol_id', 'smiles']
Saved tox21_nr_data.parquet


In [5]:
if tox21_df is not None and len(tox21_df) > 0 and "smiles" in tox21_df.columns:
    tox21_smiles_col = "smiles"
    tox21_avail = [t for t in TOX21_NR_TASKS if t in tox21_df.columns]
    print(f"Available Tox21 NR tasks: {tox21_avail}")

    LGBM_CLS = dict(n_estimators=300, num_leaves=31, learning_rate=0.1,
                    min_child_samples=5, subsample=0.8, colsample_bytree=0.8,
                    random_state=SEED, verbose=-1, n_jobs=4)

    tox21_bio_tr = np.zeros((len(tr), len(tox21_avail)), dtype=np.float32)
    tox21_bio_te = np.zeros((len(te), len(tox21_avail)), dtype=np.float32)

    for j, task in enumerate(tox21_avail):
        sub = tox21_df[[tox21_smiles_col, task]].dropna()
        sub = sub[sub[task].isin([0.0, 1.0])]
        if len(sub) < 100: continue
        X_t = impute(combined(sub[tox21_smiles_col].tolist()))
        y_t = sub[task].values.astype(int)
        m_cls = lgb.LGBMClassifier(**LGBM_CLS)
        m_cls.fit(X_t, y_t)
        tox21_bio_tr[:, j] = m_cls.predict_proba(X_tr)[:, 1]
        tox21_bio_te[:, j] = m_cls.predict_proba(X_te)[:, 1]
        print(f"  {task}: {len(sub):,} cmpds  P(active|PXR_train)={tox21_bio_tr[:,j].mean():.3f}", flush=True)

    print(f"\nTox21 bio-FP: {tox21_bio_tr.shape}")
else:
    print("Tox21 not available — using zeros placeholder")
    tox21_avail = []
    tox21_bio_tr = np.zeros((len(tr), 0), dtype=np.float32)
    tox21_bio_te = np.zeros((len(te), 0), dtype=np.float32)

# Load ChEMBL bio-FP from nb87 if available
bio_fp_tr_path = DATA_PROCESSED/"bio_fp_tr.npy"
bio_fp_te_path = DATA_PROCESSED/"bio_fp_te.npy"
if bio_fp_tr_path.exists():
    chembl_bio_tr = np.load(bio_fp_tr_path)
    chembl_bio_te = np.load(bio_fp_te_path)
    print(f"Loaded ChEMBL bio-FP: {chembl_bio_tr.shape}")
else:
    chembl_bio_tr = np.zeros((len(tr), 0), dtype=np.float32)
    chembl_bio_te = np.zeros((len(te), 0), dtype=np.float32)
    print("ChEMBL bio-FP not found — run nb87 first")

# Full biological fingerprint = ChEMBL NR + Tox21 NR
all_bio_tr = np.hstack([chembl_bio_tr, tox21_bio_tr]).astype(np.float32)
all_bio_te = np.hstack([chembl_bio_te, tox21_bio_te]).astype(np.float32)
print(f"Combined bio-FP dim: {all_bio_tr.shape[1]}")


Available Tox21 NR tasks: ['NR-AhR', 'NR-AR', 'NR-AR-LBD', 'NR-ER', 'NR-ER-LBD', 'NR-PPAR-gamma']


[19:34:42] WARNING: not removing hydrogen atom without neighbors


[19:34:43] WARNING: not removing hydrogen atom without neighbors


  NR-AhR: 6,549 cmpds  P(active|PXR_train)=0.108


[19:35:32] WARNING: not removing hydrogen atom without neighbors


...

  NR-ER-LBD: 6,955 cmpds  P(active|PXR_train)=0.005


[19:39:00] WARNING: not removing hydrogen atom without neighbors


[19:39:01] WARNING: not removing hydrogen atom without neighbors


  NR-PPAR-gamma: 6,450 cmpds  P(active|PXR_train)=0.004



Tox21 bio-FP: (4139, 6)
Loaded ChEMBL bio-FP: (4139, 5)
Combined bio-FP dim: 11


In [6]:
if all_bio_tr.shape[1] > 0:
    X_full_tr = np.hstack([X_tr, all_bio_tr])
    X_full_te = np.hstack([X_te, all_bio_te])

    oof = np.full(len(y_tr), np.nan)
    for fold, (tr_idx, va_idx) in enumerate(splits):
        m = lgb.train(LGBM, lgb.Dataset(X_full_tr[tr_idx], label=y_tr[tr_idx]),
                      valid_sets=[lgb.Dataset(X_full_tr[va_idx], label=y_tr[va_idx])],
                      callbacks=[lgb.early_stopping(50,verbose=False), lgb.log_evaluation(-1)])
        oof[va_idx] = m.predict(X_full_tr[va_idx])
        print(f"  fold {fold+1}  RAE={rae(y_tr[va_idx], oof[va_idx]):.4f}", flush=True)

    m_res = full_metrics(y_tr, oof, cliff_pairs, "combined+all_bio_fp")
    print("\n" + pd.DataFrame([m_res], index=["all_bio_fp"]).round(4).to_string())

    m_final = lgb.train(LGBM, lgb.Dataset(X_full_tr, label=y_tr), callbacks=[lgb.log_evaluation(-1)])
    te_preds = np.clip(m_final.predict(X_full_te), y_tr.min()-0.5, y_tr.max()+0.5)
else:
    print("No bio-FPs available — falling back to combined only")
    oof = np.full(len(y_tr), np.nan)
    for fold, (tr_idx, va_idx) in enumerate(splits):
        m = lgb.train(LGBM, lgb.Dataset(X_tr[tr_idx], label=y_tr[tr_idx]),
                      valid_sets=[lgb.Dataset(X_tr[va_idx], label=y_tr[va_idx])],
                      callbacks=[lgb.early_stopping(50,verbose=False), lgb.log_evaluation(-1)])
        oof[va_idx] = m.predict(X_tr[va_idx])
    m_final = lgb.train(LGBM, lgb.Dataset(X_tr, label=y_tr), callbacks=[lgb.log_evaluation(-1)])
    te_preds = np.clip(m_final.predict(X_te), y_tr.min()-0.5, y_tr.max()+0.5)

np.save(DATA_PROCESSED/"tox21_bio_tr.npy", tox21_bio_tr)
np.save(DATA_PROCESSED/"tox21_bio_te.npy", tox21_bio_te)
np.save(DATA_PROCESSED/"oof_tox21_bio_fp.npy", oof)
np.save(DATA_PROCESSED/"te_oof_tox21_bio_fp.npy", te_preds)
sub = pd.DataFrame({"Molecule Name": te["name"].values, "pEC50": te_preds})
assert len(sub)==513 and sub["pEC50"].notna().all()
p = SUBMISSIONS/"90_tox21_bio_fp.csv"; sub.to_csv(p, index=False)
print(f"Saved {p}  Test: min={te_preds.min():.2f} med={np.median(te_preds):.2f} max={te_preds.max():.2f}")


  fold 1  RAE=0.4908


  fold 2  RAE=0.5775


  fold 3  RAE=0.5995


  fold 4  RAE=0.5719


  fold 5  RAE=0.6052


  [combined+all_bio_fp] RAE=0.5639 MAE=0.5131 R²=0.6007 r=0.7750 ρ=0.7294 τ=0.5370

               RAE     MAE      R2  Pearson  Spearman  Kendall
all_bio_fp  0.5639  0.5131  0.6007    0.775    0.7294    0.537


Saved D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\90_tox21_bio_fp.csv  Test: min=2.20 med=4.96 max=6.01
